# BM25 against TF-IDF in NumPy

A ferry company's help desk retrieves from fifteen notes. One of them is a news item that
says "ferry" eight times, and TF-IDF puts it first for many questions that mention the
ferry. BM25, the sparse score the lesson mentions, also starts from word counts and rare
words, but it gives each repeat of a word less extra weight and adjusts for the length of
each note. You will implement BM25, compare the two retrievers by recall@k on twelve test
questions, some of which have two notes that answer them, and find how many repeats of
one word a spam note needs to come first under each score.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import re
import numpy as np

notes = [
    "Tickets can be bought online or at the kiosk in the terminal.",                                   # 0
    "A one way ticket costs 9 dollars for adults and 4 dollars for children.",                        # 1
    "A monthly pass gives unlimited rides for 120 dollars, cheaper than 14 tickets.",                 # 2
    "Bikes ride free on the lower deck, and the crew will help you tie a bike down.",                 # 3
    "Dogs on a leash are welcome on the outer deck. A dog must stay off the seats.",                  # 4
    "Cars must check in 30 minutes before departure. Car lanes close 10 minutes before departure.",   # 5
    "The first ferry leaves the island at 6 am, and the last ferry leaves the city at 11 pm.",        # 6
    "If a sailing is canceled for fog or high wind, every ticket for that sailing is refunded.",     # 7
    "A refund for an unused ticket is paid to the original card within 5 days.",                      # 8
    "Lost items are kept at the terminal office for 30 days. Call the office to ask about a lost bag.",  # 9
    "Wheelchair users board by the ramp to the lower deck. Ask the crew for help with the ramp.",     # 10
    "The cafe on the upper deck sells coffee, sandwiches and soup.",                                  # 11
    "Parking at the terminal costs 12 dollars per day, and the car park opens at 5 am.",              # 12
    "Children under 5 ride free, and a child under 12 must travel with an adult.",                    # 13
    "Ferry news: the ferry welcomes you aboard the fastest ferry on the bay, and the ferry crew "
    "wishes you a pleasant crossing on the ferry. Follow the ferry online for ferry photos and ferry stories.",  # 14
]
questions = [                                        # (question, the notes that answer it)
    ("How do I get a refund for my ticket?", [8, 7]),
    ("Can I bring my bike on the ferry?", [3]),
    ("When does the last ferry leave?", [6]),
    ("How much is a ticket for a child?", [1, 13]),
    ("Where can I buy tickets?", [0]),
    ("Can I get coffee on the ferry?", [11]),
    ("What happens to my ticket if a sailing is canceled for wind?", [7, 8]),
    ("Can my dog come on the ferry?", [4]),
    ("Where do I park my car?", [12, 5]),
    ("I lost my bag on the ferry", [9]),
    ("Can a wheelchair board the ferry?", [10]),
    ("Is a monthly pass cheaper than tickets?", [2, 1]),
]

tokenize = lambda s: re.findall(r"[a-z0-9]+", s.lower())

def count_matrix(texts):
    """The sorted vocabulary of the texts, and how often each word appears in each text."""
    vocab = sorted({w for t in texts for w in tokenize(t)})
    col = {w: j for j, w in enumerate(vocab)}
    tf = np.zeros((len(texts), len(vocab)))
    for i, t in enumerate(texts):
        for w in tokenize(t):
            tf[i, col[w]] += 1
    return vocab, tf

def count_questions(texts, vocab):
    """Word counts of questions over a fixed vocabulary. Words outside it are dropped."""
    col = {w: j for j, w in enumerate(vocab)}
    Qc = np.zeros((len(texts), len(vocab)))
    for i, t in enumerate(texts):
        for w in tokenize(t):
            if w in col:
                Qc[i, col[w]] += 1
    return Qc

def tfidf_scores(tf, Qc):
    """Cosine of TF-IDF vectors, as in the lesson: one row per question, one column per note."""
    idf = np.log(len(tf) / (tf > 0).sum(axis=0)) + 1
    D = tf * idf
    D = D / np.linalg.norm(D, axis=1, keepdims=True)
    Q = np.atleast_2d(Qc) * idf
    lengths = np.linalg.norm(Q, axis=1, keepdims=True)
    Q = np.divide(Q, lengths, out=np.zeros_like(Q), where=lengths > 0)
    return Q @ D.T

def bm25_scores(tf, Qc, k1=1.5, b=0.75):
    """BM25 scores: each question adds up the weights of its distinct words (Part 1)."""
    return (np.atleast_2d(Qc) > 0) @ bm25_weights(tf, k1, b).T

vocab, tf = count_matrix(notes)
Qc = count_questions([q for q, _ in questions], vocab)
relevant = [r for _, r in questions]
print(f"{len(notes)} notes, {len(vocab)} words, {len(questions)} test questions")
print("note lengths:", tf.sum(axis=1).astype(int))

## Part 1 (coding task): BM25 weights

BM25 scores a note for a question by adding up one weight for each distinct word of the
question. The weight of word $w$ in note $d$ is

$$
W_{dw} = \mathrm{idf}(w)\,\frac{f_{dw}\,(k_1 + 1)}{f_{dw} + k_1\left(1 - b + b\,L_d / \bar L\right)},
\qquad
\mathrm{idf}(w) = \ln\left(1 + \frac{n - \mathrm{df}(w) + 0.5}{\mathrm{df}(w) + 0.5}\right).
$$

Here $f_{dw}$ is the number of times $w$ appears in note $d$, $L_d$ is the length of note
$d$ (its number of words, repeats included), $\bar L$ is the mean length of the $n$ notes,
and $\mathrm{df}(w)$ is the number of notes that contain $w$. The usual settings are
$k_1 = 1.5$ and $b = 0.75$.

Write `bm25_weights(tf, k1=1.5, b=0.75)`. `tf` is a count matrix with one row per note and
one column per word. Return the matrix of weights $W$, of the same shape, using
whole-array operations. The setup's `bm25_scores` uses it.

In [ ]:
def bm25_weights(tf, k1=1.5, b=0.75):
    """The BM25 weight of every word in every note."""
    tf = np.asarray(tf, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
tf_small = np.array([[2, 0, 1, 0],
                     [0, 1, 0, 0],
                     [1, 1, 3, 1]])
W0 = np.asarray(bm25_weights(tf_small, k1=1.5, b=0))
assert W0.shape == (3, 4), f"expected one row per note and one column per word, shape (3, 4), got {W0.shape}"
assert np.all(W0[tf_small == 0] == 0), "a word that is not in a note must have weight 0 there"
assert (W0 >= 0).all(), "a weight came out negative, but ln(1 + ...) is positive for every word: check the idf formula and df(w), the number of notes that contain w"
assert np.allclose(W0, [[0.671434, 0, 0.470004, 0],
                        [0, 0.470004, 0, 0],
                        [0.470004, 0.470004, 0.783339, 0.980829]], atol=1e-6), \
    f"wrong weights with b = 0, where the lengths drop out: check idf and the count fraction\n{np.round(W0, 6)}"
W = np.asarray(bm25_weights(tf_small))
assert not np.allclose(W, [[0.703749, 0, 0.502294, 0],
                           [0, 0.632697, 0, 0],
                           [0.355678, 0.355678, 0.664652, 0.742249]], atol=1e-6), \
    "these lengths count distinct words: L_d counts every word of the note, repeats included"
assert np.allclose(W, [[0.693732, 0, 0.49215, 0],
                       [0, 0.686137, 0, 0],
                       [0.345591, 0.345591, 0.652783, 0.721198]], atol=1e-6), \
    f"wrong weights with b = 0.75: check L_d and the mean length over the notes\n{np.round(W, 6)}"
assert np.allclose(bm25_weights(tf_small, k1=1.2, b=1.0)[1, 1], 0.7603, atol=1e-4), "k1 and b must be used as passed"

w = np.asarray(bm25_weights(tf))[14, vocab.index("ferry")]
print(f'the news note says "ferry" {int(tf[14, vocab.index("ferry")])} times: BM25 weight {w:.3f}')
print("Part 1 passed")

## Part 2 (coding task): Recall@k with several answer notes

Write `recall_at_k(S, relevant, k)`. `S` holds scores, one row per question and one column
per note, and `relevant[i]` lists the notes that answer question `i`. Each question
retrieves the `k` notes with the highest scores, and a tie goes to the note that comes
first in the list of notes. A question's recall@k is the fraction of its answer notes
that are among the `k` it retrieves, and the recall@k of the test set is the mean of that
over the questions.

In [ ]:
def recall_at_k(S, relevant, k):
    """Mean over the questions of the share of their answer notes in their top k."""
    S = np.asarray(S, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
S2 = np.array([[0.9, 0.1, 0.8, 0.3],
               [0.2, 0.6, 0.6, 0.1],
               [0.5, 0.4, 0.3, 0.2]])
rel2 = [[0, 1, 2], [2], [3]]
r2 = recall_at_k(S2, rel2, 2)
assert not np.isclose(r2, 0.6), "this pools the answer notes of all questions (3 found of 5): average the share of each question instead"
assert not np.isclose(r2, 0.5), "this divides by k, which is precision@k: divide by the number of answer notes of each question"
assert np.isclose(r2, 5 / 9), f"expected recall@2 = (2/3 + 1 + 0) / 3 = 0.556, got {r2}"
r1 = recall_at_k(S2, rel2, 1)
assert not np.isclose(r1, 4 / 9), "in row 1 notes 1 and 2 tie, and the tie goes to note 1, which comes first"
assert np.isclose(r1, 1 / 9), f"expected recall@1 = (1/3 + 0 + 0) / 3 = 0.111, got {r1}"
assert np.isclose(recall_at_k(S2, rel2, 4), 1.0), "with k equal to the number of notes every answer note is retrieved"

St, Sb = tfidf_scores(tf, Qc), bm25_scores(tf, Qc)
print("k   TF-IDF   BM25")
for k in (1, 2, 3):
    print(f"{k}   {recall_at_k(St, relevant, k):6.3f}   {recall_at_k(Sb, relevant, k):5.3f}")
print("news note first for", int(np.sum(St.argmax(axis=1) == 14)), "questions under TF-IDF and",
      int(np.sum(Sb.argmax(axis=1) == 14)), "under BM25")
print("Part 2 passed")

## Part 3 (coding task): How many repeats a spam note needs

A spammer adds one note to the collection: the words "Cheap ferry tickets" followed by
one word repeated $f$ times, for example "Cheap ferry tickets refund refund refund" for
$f = 3$. Write `repeats_to_rank_first(score_fn, question, word, max_repeats=200)`. It
returns the smallest $f$ from 1 to `max_repeats` for which the spam note scores strictly
higher than every other note for `question`, or `None` if no such $f$ exists. `score_fn`
is `tfidf_scores` or `bm25_scores` from the setup.

In [ ]:
SPAM = "Cheap ferry tickets"

def repeats_to_rank_first(score_fn, question, word, max_repeats=200):
    """Smallest number of repeats of word that puts the spam note first, or None."""
    # TODO
    raise NotImplementedError

In [ ]:
q_refund = "How do I get a refund for my ticket?"
ft = repeats_to_rank_first(tfidf_scores, q_refund, "refund")
assert ft is not None, "TF-IDF does put the spam note first for some number of repeats of 'refund'"
assert ft != 1, "got 1, but one 'refund' is not enough under TF-IDF: return the first f that wins, counting from 1, and score the spam note with all of its words as a note of the collection"
assert ft == 2, f"expected 2 repeats under TF-IDF, got {ft}"
assert repeats_to_rank_first(tfidf_scores, q_refund, "refund", max_repeats=1) is None, "with max_repeats = 1 no f wins: return None"
fb = repeats_to_rank_first(bm25_scores, q_refund, "refund")
assert fb is None, f"under BM25 no number of repeats of 'refund' up to 200 puts the spam note first, got {fb}"
fp = repeats_to_rank_first(bm25_scores, "Where do I park my car?", "park")
assert fp == 3, f"for 'Where do I park my car?' BM25 needs 3 repeats of 'park', got {fp}"
tie = lambda counts, Qc: np.zeros((len(np.atleast_2d(Qc)), len(counts)))   # every note scores 0
assert repeats_to_rank_first(tie, q_refund, "refund", max_repeats=3) is None, "every note scores 0 here, so the spam note only ties the others: a tie is not a strict win"

print(" f   TF-IDF spam  best real   BM25 spam  best real")
for f in (1, 2, 5, 20, 200):
    voc, counts = count_matrix(notes + [SPAM + " refund" * f])
    qv = count_questions([q_refund], voc)
    st, sb = tfidf_scores(counts, qv)[0], bm25_scores(counts, qv)[0]
    print(f"{f:3d}   {st[-1]:11.3f}  {st[:-1].max():9.3f}   {sb[-1]:9.3f}  {sb[:-1].max():9.3f}")
print("Part 3 passed")

## Solution

### Part 1: BM25 weights

In [ ]:
def bm25_weights(tf, k1=1.5, b=0.75):
    """The BM25 weight of every word in every note."""
    tf = np.asarray(tf, dtype=float)
    n = len(tf)
    df = (tf > 0).sum(axis=0)                               # notes that contain each word
    idf = np.log(1 + (n - df + 0.5) / (df + 0.5))
    length = tf.sum(axis=1)                                 # words per note, repeats included
    norm = k1 * (1 - b + b * length / length.mean())        # one value per note
    return idf * tf * (k1 + 1) / (tf + norm[:, None])

**Why it works.** Each piece is one array operation. `df` counts the notes with a nonzero count in each
column, so `idf` is a row of length $V$. The lengths and the length factor
$k_1(1 - b + b L_d / \bar L)$ are one value per note, and `norm[:, None]` turns them into
a column, so that broadcasting matches every row of `tf` with its own note's length and
every column with its own word's idf.

The fraction $f(k_1 + 1)/(f + K)$ is what limits repeats. It is 1 when $f = 1$ and
$K = k_1$, and it can never pass $k_1 + 1 = 2.5$ however large $f$ grows. The eight
"ferry" of the 35-word news note weigh 3.47, less than two notes of average length that
say "ferry" once, at 1.86 each. With $b > 0$, a note
longer than average has a larger $K$, so each of its words weighs less. With $b = 0$ the
lengths drop out, which is why the check tests the idf there first.

### Part 2: Recall@k with several answer notes

In [ ]:
def recall_at_k(S, relevant, k):
    """Mean over the questions of the share of their answer notes in their top k."""
    S = np.asarray(S, dtype=float)
    top = np.argsort(-S, axis=1, kind="stable")[:, :k]     # a stable sort keeps ties in note order
    shares = [np.isin(r, t).mean() for t, r in zip(top, relevant)]
    return float(np.mean(shares))

**Why it works.** `np.argsort(-S, axis=1, kind="stable")` sorts every row from the highest score down, and a
stable sort keeps tied notes in their original order, so a tie goes to the note listed
first. Reversing an ascending sort, `np.argsort(S)[:, ::-1]`, would reverse the ties too.
`np.isin(r, t)` marks which answer notes made the top `k`, and its mean is that
question's share. Averaging the shares gives every question the same weight. Pooling,
answer notes found over answer notes in total, would let a question with two answer
notes count twice as much as one with a single answer note.

On the ferry notes, TF-IDF puts the news note first for six of the twelve questions and
BM25 for three, and recall@1 rises from 0.25 to 0.50. A question with two answer notes
can reach at most 0.5 at k = 1, so the best possible recall@1 on this set is
(7 + 5 × 0.5) / 12 = 0.79. At k = 2 the two retrievers tie: wherever either one puts the
news note first, the answer note comes second.

### Part 3: How many repeats a spam note needs

In [ ]:
SPAM = "Cheap ferry tickets"

def repeats_to_rank_first(score_fn, question, word, max_repeats=200):
    """Smallest number of repeats of word that puts the spam note first, or None."""
    for f in range(1, max_repeats + 1):
        texts = notes + [SPAM + (" " + word) * f]
        voc, counts = count_matrix(texts)                  # the spam note changes df, n and the mean length
        s = score_fn(counts, count_questions([question], voc))[0]
        if s[-1] > s[:-1].max():
            return f
    return None

**Why it works.** The spam note joins the collection, so each value of $f$ needs a new count matrix: the
spam note changes the vocabulary, the number of notes, the document frequency of the
repeated word and, for BM25, the mean length. Comparing the last score with the maximum
of the others asks for a strict win, so a tie with a real note does not count.

The table shows why the two scores behave so differently. Under TF-IDF the spam vector
turns toward the "refund" axis as $f$ grows, and its cosine with the question approaches
the share of the question's vector along "refund", 0.65 here, well above the 0.40 of the
real refund note. Two repeats already win. Under BM25 each repeat adds less than the one
before, the spam note grows longer with every repeat, and its score creeps toward the
ceiling $\mathrm{idf} \times (k_1 + 1)$ without reaching the real note, whose score
even rises, since the long spam note raises the mean length and makes every real note
relatively short.

BM25 does not make spam impossible. "Where do I park my car?" falls to three repeats of
"park", because the spam note is short and "park" is a rare word. What BM25 removes is
the easy route of repeating one word until it drowns everything else.